# mtrl on Colab: smoke test

Checks a fresh runtime end to end: setup, unit tests, a few SFT and RL steps with
Qwen3.5-2B on en-is and en-kk, and resume from a checkpoint. About 15–25 minutes, mostly
installs and the first model download. Pick **L4** (or A100) under Runtime → Change
runtime type, then Run all. To run real experiments use `setup/colab_run.ipynb`.

In [ ]:
REF = 'train/merge-ym'   # branch or commit to test

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
for d in ('runs', 'ckpt', 'data_cache'):
    os.makedirs(f'/content/drive/MyDrive/mtrl/{d}', exist_ok=True)

%cd /content
if not os.path.isdir('/content/mtrl/.git'):
    !git clone -q https://github.com/ETHz-DL-team29/mtrl.git /content/mtrl
!git -C /content/mtrl fetch -q && git -C /content/mtrl checkout -q {REF} && (git -C /content/mtrl pull -q --ff-only 2>/dev/null || true)
%cd /content/mtrl
!git log --oneline -1

In [ ]:
!bash setup/colab_setup.sh
import sys
sys.path.insert(0, '/content/mtrl/src')   # this kernel started before mtrl was installed
os.environ['MTRL_ENV'] = open('.env').read().split('=', 1)[1].strip()
print('MTRL_ENV =', os.environ['MTRL_ENV'])

## Unit tests

In [ ]:
!python -m pytest -q tests 2>&1 | tail -3

## SFT and RL smoke runs (6 steps each), starting from scratch

In [ ]:
!rm -rf /content/drive/MyDrive/mtrl/runs/quick_sft /content/drive/MyDrive/mtrl/runs/quick_rl /content/drive/MyDrive/mtrl/ckpt/quick_sft /content/drive/MyDrive/mtrl/ckpt/quick_rl
!python scripts/run_experiments.py configs/train/quick_sft.yaml configs/train/quick_rl.yaml --no-eval --data-cache /content/drive/MyDrive/mtrl/data_cache

## Resume: raise the step count; training must continue at step 6, not 0

In [ ]:
!sed -i 's/steps: 6/steps: 8/' configs/train/quick_sft.yaml
!python scripts/train.py configs/train/quick_sft.yaml 2>&1 | grep -E "resumed|^\[sft\]" | cut -c1-160
!git checkout -q configs/train/quick_sft.yaml

## Summary to report back

In [ ]:
import json, pathlib
runs = pathlib.Path('/content/drive/MyDrive/mtrl/runs')
for r in ['quick_sft', 'quick_rl']:
    log = [json.loads(l) for l in (runs / r / 'train_log.jsonl').read_text().splitlines()]
    val = [json.loads(l) for l in (runs / r / 'val_log.jsonl').read_text().splitlines()] if (runs / r / 'val_log.jsonl').exists() else []
    print(r, '| steps', len(log), '| max vram_gb', max(x.get('vram_gb', 0) for x in log),
          '| median sec/step', sorted(x['sec'] for x in log[1:])[len(log[1:]) // 2] if len(log) > 1 else None,
          '| nonfinite', sum(x.get('skipped_nonfinite', 0) for x in log),
          '| last val', {k: v for k, v in (val[-1] if val else {}).items() if k in ('val_chrf', 'val_gate_pass', 'error')})